<figure>
<center>
<img src="https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png"/>
</center></figure>

# **Universidad de Buenos Aires**
## **Facultad de Ciencias Económicas**
### **Métodos Predictivos**
### Cátedra: Bianco
#### **Cross-Validation y Optimización**

## Ensambles

In [1]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, GridSearchCV
import numpy as np
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

In [2]:
# Descarga directa desde un repositorio verificado con la versión ampliada de 2500 registros
url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df_base = pd.read_csv(url)

df = df_base.copy()

# # Renombramos columnas
df.columns = ['Edad', 'Sexo', 'IMC', 'Hijos', 'Fumador', 'Region', 'Cargos_Medicos']

# Convertimos las variables categóricas de texto a numéricas (0 y 1)
df = pd.get_dummies(df, columns=['Sexo', 'Fumador'], drop_first=True, dtype=int)
df = pd.get_dummies(df, columns=['Region'], drop_first=False, dtype=int)


print("\n--- Estructura del Dataset de Seguros para Ensambles ---")
print(f"Dimensiones reales: {df.shape[0]} filas x {df.shape[1]} columns")
print("\nPrimeros registros (Base 100% numérica lista para modelar):")
df.head()


--- Estructura del Dataset de Seguros para Ensambles ---
Dimensiones reales: 1338 filas x 10 columns

Primeros registros (Base 100% numérica lista para modelar):


,Edad,IMC,Hijos,Cargos_Medicos,Sexo_male,Fumador_yes,Region_northeast,Region_northwest,Region_southeast,Region_southwest
0,19,27.900,0,16884.92400,0,1,0,0,0,1
1,18,33.770,1,1725.55230,1,0,0,0,1,0
2,28,33.000,3,4449.46200,1,0,0,0,1,0
3,33,22.705,0,21984.47061,1,0,0,1,0,0
4,32,28.880,0,3866.85520,1,0,0,1,0,0


In [3]:
# Separamos las características (X) de la variable objetivo (y)
# Cargos_Medicos' es lo que queremos predecir. El resto son los predictores.
X = df.drop(columns=['Cargos_Medicos'])
y = df['Cargos_Medicos']

# Hacemos el Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Total de registros en la base: {df.shape[0]}")
print(f"Registros para ENTRENAR los modelos (X_train): {X_train.shape[0]}")
print(f"Registros para TESTEAR los modelos (X_test): {X_test.shape[0]}")
print(f"Cantidad de variables predictoras: {X_train.shape[1]}")

Total de registros en la base: 1338
Registros para ENTRENAR los modelos (X_train): 1070
Registros para TESTEAR los modelos (X_test): 268
Cantidad de variables predictoras: 9


### Cross-Validation

In [4]:
# Definimos un Random Forest base (con parámetros por defecto) para evaluar
modelo_base = RandomForestRegressor(random_state=42)

# cross_val_score divide X_train en k folds y evalúa el modelo k veces
# cv=5     → cantidad de folds (5 particiones del train set)
# scoring  → métrica de evaluación (neg_mean_squared_error = negativo del MSE,
#            sklearn lo devuelve negativo por convención de "mayor es mejor")
cv_scores = cross_val_score(
    modelo_base,
    X_train,
    y_train,
    cv=5,                                 # Número de folds
    scoring='neg_mean_squared_error'      # Métrica: MSE negativo
)

In [5]:
# Convertimos a RMSE positivo para que sea más interpretable
cv_rmse = np.sqrt(-cv_scores)

print("=== Resultados de Cross-Validation (modelo base) ===")
print(f"RMSE por fold: {cv_rmse.round(2)}")
print(f"RMSE promedio: {cv_rmse.mean():.2f}")     # Error esperado del modelo
print(f"Desvío estándar: {cv_rmse.std():.2f}")    # Varianza entre folds

=== Resultados de Cross-Validation (modelo base) ===
RMSE por fold: [5075.71 3982.34 5087.11 5395.57 4987.72]
RMSE promedio: 4905.69
Desvío estándar: 481.89


### Optimización de Parámetros con Grid Search

In [6]:
# Grilla de hiperparámetros a explorar
# Cada clave corresponde a un parámetro del RandomForestRegressor
param_grid = {
    'n_estimators':      [100, 300, 500],   # Cantidad de árboles en el bosque
    'max_depth':         [3, 5, 7, 10],     # Profundidad máxima de cada árbol
    'min_samples_leaf':  [1, 3, 5],         # Mínimo de muestras en una hoja final
    'min_samples_split': [2, 4, 6]          # Mínimo de muestras para dividir un nodo
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,                                   # Folds para evaluar cada combinación
    scoring='neg_mean_squared_error',       # Misma métrica que el CV anterior
    n_jobs=-1,                              # Paraleliza el entrenamiento
    verbose=1                               # Muestra progreso en consola
)

grid_search.fit(X_train, y_train)

print("\n=== Resultados de Grid Search ===")
print(f"Mejores hiperparámetros: {grid_search.best_params_}")
print(f"Mejor RMSE en CV: {np.sqrt(-grid_search.best_score_):.2f}")

Fitting 5 folds for each of 108 candidates, totalling 540 fits

=== Resultados de Grid Search ===
Mejores hiperparámetros: {'max_depth': 5, 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 100}
Mejor RMSE en CV: 4649.99


In [7]:
# Reentrenamiento con los mejores parámetros sobre todo X_train
mejor_modelo = RandomForestRegressor(**grid_search.best_params_, random_state=42)
mejor_modelo.fit(X_train, y_train)

y_pred = mejor_modelo.predict(X_test)

In [8]:
# Predicciones sobre el conjunto de test con el mejor modelo encontrado
y_pred = grid_search.predict(X_test)

# Métricas de evaluación
rmse = np.sqrt(mean_squared_error(y_test, y_pred))  # Error cuadrático medio (raíz)
mae  = mean_absolute_error(y_test, y_pred)           # Error absoluto medio
r2   = r2_score(y_test, y_pred)                      # Coeficiente de determinación

print("=== Métricas finales sobre Test Set ===")
print(f"RMSE : {rmse:.2f}")   # En las mismas unidades que Cargos_Medicos
print(f"MAE  : {mae:.2f}")    # Más robusto ante outliers que el RMSE
print(f"R²   : {r2:.4f}")     # Proporción de varianza explicada (1.0 = perfecto)

=== Métricas finales sobre Test Set ===
RMSE : 4346.76
MAE  : 2483.90
R²   : 0.8783


In [9]:
# Comparación rápida CV vs Test para detectar overfitting
print("\n=== Comparación CV vs Test ===")
print(f"RMSE promedio en CV : {cv_rmse.mean():.2f}")
print(f"RMSE final en Test  : {rmse:.2f}")


=== Comparación CV vs Test ===
RMSE promedio en CV : 4905.69
RMSE final en Test  : 4346.76


### Random Search

In [10]:
# Grilla de hiperparámetros a explorar (misma que en Grid Search)
param_grid_random = {
    'n_estimators':      [100, 300, 500],   # Cantidad de árboles en el bosque
    'max_depth':         [3, 5, 7, 10],     # Profundidad máxima de cada árbol
    'min_samples_leaf':  [1, 3, 5],         # Mínimo de muestras en una hoja final
    'min_samples_split': [2, 4, 6]          # Mínimo de muestras para dividir un nodo
}

random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_grid_random,
    n_iter=20,                              # Cantidad de combinaciones a probar
    cv=5,                                   # Folds para evaluar cada combinación
    scoring='neg_mean_squared_error',       # Misma métrica que el CV y Grid Search
    n_jobs=-1,                              # Paraleliza el entrenamiento
    random_state=81,                        # Reproducibilidad del muestreo aleatorio
    verbose=1                               # Muestra progreso en consola
)

random_search.fit(X_train, y_train)

print("=== Resultados de Random Search ===")
print(f"Mejores hiperparámetros: {random_search.best_params_}")
print(f"Mejor RMSE en CV: {np.sqrt(-random_search.best_score_):.2f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
=== Resultados de Random Search ===
Mejores hiperparámetros: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_depth': 5}
Mejor RMSE en CV: 4652.72


In [11]:
# Reentrenamiento explícito con los mejores parámetros sobre todo X_train
mejor_modelo_random = RandomForestRegressor(**random_search.best_params_, random_state=42)
mejor_modelo_random.fit(X_train, y_train)

y_pred_random = mejor_modelo_random.predict(X_test)

In [12]:
rmse_random = np.sqrt(mean_squared_error(y_test, y_pred_random))
mae_random  = mean_absolute_error(y_test, y_pred_random)
r2_random   = r2_score(y_test, y_pred_random)

print("\n=== Métricas finales sobre Test Set (Random Search) ===")
print(f"RMSE : {rmse_random:.2f}")
print(f"MAE  : {mae_random:.2f}")
print(f"R²   : {r2_random:.4f}")


=== Métricas finales sobre Test Set (Random Search) ===
RMSE : 4331.69
MAE  : 2460.40
R²   : 0.8791


### Bayesian Search

In [13]:
param_grid_bayes = {
    'n_estimators':      Integer(100, 500),         # Cantidad de árboles en el bosque
    'max_depth':         Integer(3, 10),             # Profundidad máxima de cada árbol
    'min_samples_leaf':  Integer(1, 5),              # Mínimo de muestras en una hoja final
    'min_samples_split': Integer(2, 6)               # Mínimo de muestras para dividir un nodo
}

# BayesSearchCV usa un modelo probabilístico (Gaussian Process) para estimar
# qué zonas de la grilla tienen más chances de mejorar el resultado
# n_iter=20 → cantidad de combinaciones a evaluar
# random_state=42 → reproducibilidad
bayes_search = BayesSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    search_spaces=param_grid_bayes,
    n_iter=20,                              # Iteraciones de búsqueda bayesiana
    cv=5,                                   # Folds para evaluar cada combinación
    scoring='neg_mean_squared_error',       # Misma métrica que los anteriores
    n_jobs=-1,                              # Paraleliza el entrenamiento
    random_state=42,                        # Reproducibilidad
    verbose=1                               # Muestra progreso en consola
)

bayes_search.fit(X_train, y_train)

print("=== Resultados de Bayesian Search ===")
print(f"Mejores hiperparámetros: {bayes_search.best_params_}")
print(f"Mejor RMSE en CV: {np.sqrt(-bayes_search.best_score_):.2f}")

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi

In [14]:
# Reentrenamiento explícito con los mejores parámetros sobre todo X_train
mejor_modelo_bayes = RandomForestRegressor(**bayes_search.best_params_, random_state=42)
mejor_modelo_bayes.fit(X_train, y_train)

y_pred_bayes = mejor_modelo_bayes.predict(X_test)

In [15]:
rmse_bayes = np.sqrt(mean_squared_error(y_test, y_pred_bayes))
mae_bayes  = mean_absolute_error(y_test, y_pred_bayes)
r2_bayes   = r2_score(y_test, y_pred_bayes)

print("\n=== Métricas finales sobre Test Set (Bayesian Search) ===")
print(f"RMSE : {rmse_bayes:.2f}")
print(f"MAE  : {mae_bayes:.2f}")
print(f"R²   : {r2_bayes:.4f}")


=== Métricas finales sobre Test Set (Bayesian Search) ===
RMSE : 4412.65
MAE  : 2561.82
R²   : 0.8746


In [16]:
# COMPARACIÓN FINAL: GRID SEARCH vs RANDOM SEARCH vs BAYESIAN SEARCH
print("\n=== Comparación de estrategias de optimización ===")
print(f"{'':25} {'Grid Search':>13} {'Random Search':>13} {'Bayes Search':>13}")
print(f"{'Mejor RMSE en CV':25} {np.sqrt(-grid_search.best_score_):>13.2f} {np.sqrt(-random_search.best_score_):>13.2f} {np.sqrt(-bayes_search.best_score_):>13.2f}")
print(f"{'RMSE en Test':25} {rmse:>13.2f} {rmse_random:>13.2f} {rmse_bayes:>13.2f}")
print(f"{'MAE en Test':25} {mae:>13.2f} {mae_random:>13.2f} {mae_bayes:>13.2f}")
print(f"{'R² en Test':25} {r2:>13.4f} {r2_random:>13.4f} {r2_bayes:>13.4f}")
print(f"{'Combinaciones probadas':25} {108:>13} {20:>13} {20:>13}")


=== Comparación de estrategias de optimización ===
                            Grid Search Random Search  Bayes Search
Mejor RMSE en CV                4649.99       4652.72       4630.74
RMSE en Test                    4346.76       4331.69       4412.65
MAE en Test                     2483.90       2460.40       2561.82
R² en Test                       0.8783        0.8791        0.8746
Combinaciones probadas              108            20            20
